In [1]:
import sys
import os
import mysql.connector
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Ambil konfigurasi database dari folder utama project kalian
sys.path.append(os.path.abspath('..'))
from config import get_db_config

config = get_db_config()

# 1. Koneksi ke Database Baru (Fase Migrasi Sekarang)
db_new = mysql.connector.connect(**config['db_new'])
cursor_new = db_new.cursor(dictionary=True)

# 2. Koneksi ke Database Masa Depan (DB_FUTURE)
# Catatan: Pastikan di file config.py kalian sudah ada key 'db_future' ya!
db_future = mysql.connector.connect(**config['db_future'])
cursor_future = db_future.cursor(dictionary=True)

print(f"✅ Sukses Terhubung ke Database Baru : {config['db_new']['database']}")
print(f"🚀 Sukses Terhubung ke DB_FUTURE     : {config['db_future']['database']}")

✅ Sukses Terhubung ke Database Baru : 3
🚀 Sukses Terhubung ke DB_FUTURE     : 3


In [2]:
tables_to_check = [
    # --- Bagian Cimut ---
    "izin_karyawan", 
    "verifikasi_izin", 
    "absensi", 
    "verifikasi_absensi", 
    "karyawan_resign",
    
    # --- Bagian Afrida (Fokus Utama) ---
    "jadwal", 
    "jadwal_hari", 
    "jadwal_detail", 
    "jadwal_pengajar", 
    "jadwal_siswa", 
    "catatan_kelas", 
    "catatan_kelas_tag", 
    "catatan_mingguan",
    
    # --- Bagian Hanif (Fokus Utama) ---
    "siswa", 
    "kursus_siswa", 
    "siswa_keluar", 
    "mitra", 
    "mitra_progres", 
    "kemitraan_verifikator", 
    "siswa_mitra", 
    "siswa_mitra_keluar"
]

In [3]:
# === Cell 2: Inspeksi Detektor Pintar dengan Prioritas Target Revisi di Atas ===
import pandas as pd
import numpy as np

print("================================================================================")
print(" 🕵️‍♂️ FORENSIK ARSITEKTUR KONTROL: DETEKTOR AUTO-SMART COMPILER (REVISI PRIORITY) 🕵️‍♂️ ")
print("================================================================================")

# List penampung data di memori untuk keperluan sorting visualisasi
revisi_tables_queue = []
identical_tables_queue = []

# --- TAHAP A: PROSES PEN ARIKAN DATA & EVALUASI STRUKTUR DI BELAKANG LAYAR ---
for table in tables_to_check:
    try:
        # 1. Ambil data asli dari DB_NEW untuk kebutuhan .info() dan sampel isi data
        query = f"SELECT * FROM `{table}`"
        df_real_data = pd.read_sql(query, db_new)
        
        # 2. Tarik Struktur Fisik Kolom dari DB_NEW
        query_new_struct = f"""
        SELECT 
            c.COLUMN_NAME AS 'Nama Kolom',
            c.COLUMN_TYPE AS 'Tipe Data MySQL',
            CONCAT(
                CASE WHEN c.IS_NULLABLE = 'YES' THEN '✅ NULL (Boleh Kosong)' ELSE '🛑 NOT NULL (Wajib Isi)' END,
                CASE WHEN c.EXTRA = 'auto_increment' THEN '\\n🚀 AUTO_INCREMENT' ELSE '' END
            ) AS 'Aturan Nullability & Increment',
            CASE 
                WHEN c.COLUMN_KEY = 'PRI' THEN '🔑 PRIMARY KEY (PK)'
                WHEN c.COLUMN_KEY = 'MUL' AND k.REFERENCED_TABLE_NAME IS NOT NULL THEN '🔗 FOREIGN KEY (FK)'
                WHEN c.COLUMN_KEY = 'MUL' THEN 'INDEX'
                ELSE '-'
            END AS 'Status Kunci',
            CASE 
                WHEN k.REFERENCED_TABLE_NAME IS NOT NULL THEN CONCAT(k.REFERENCED_TABLE_NAME, ' (', k.REFERENCED_COLUMN_NAME, ')')
                ELSE '-'
            END AS 'Rujukan Induk (FK Origin)',
            CASE WHEN c.DATA_TYPE = 'enum' THEN REPLACE(REPLACE(REPLACE(c.COLUMN_TYPE, 'enum(', ''), ')', ''), "'", "") ELSE '-' END AS 'Daftar Pilihan ENUM'
        FROM INFORMATION_SCHEMA.COLUMNS c
        LEFT JOIN INFORMATION_SCHEMA.KEY_COLUMN_USAGE k 
            ON c.TABLE_SCHEMA = k.TABLE_SCHEMA AND c.TABLE_NAME = k.TABLE_NAME AND c.COLUMN_NAME = k.COLUMN_NAME
        WHERE c.TABLE_SCHEMA = '{config['db_new']['database']}' AND c.TABLE_NAME = '{table}'
        ORDER BY c.ORDINAL_POSITION;
        """
        df_struct_new = pd.read_sql(query_new_struct, db_new).drop_duplicates(subset=['Nama Kolom'], keep='first').reset_index(drop=True)
        
        # Lacak tabel anak yang nge-FK ke PK tabel ini di DB_NEW
        pk_referenced_list = []
        for idx, row_skri in df_struct_new.iterrows():
            if row_skri['Status Kunci'] == '🔑 PRIMARY KEY (PK)':
                lookup_fk_query = f"""
                SELECT CONCAT(TABLE_NAME, ' (', COLUMN_NAME, ')') as relasi
                FROM INFORMATION_SCHEMA.KEY_COLUMN_USAGE
                WHERE REFERENCED_TABLE_SCHEMA = '{config['db_new']['database']}'
                  AND REFERENCED_TABLE_NAME = '{table}'
                  AND REFERENCED_COLUMN_NAME = '{row_skri['Nama Kolom']}';
                """
                df_relasi_luar = pd.read_sql(lookup_fk_query, db_new)
                if not df_relasi_luar.empty:
                    pk_referenced_list.append("\n".join(df_relasi_luar['relasi'].tolist()))
                else:
                    pk_referenced_list.append("-")
            else:
                pk_referenced_list.append("-")
        df_struct_new['Tabel Yang nge-FK (DB_NEW)'] = pk_referenced_list

        # 3. Tarik Struktur Fisik Kolom dari DB_FUTURE
        query_future_struct = f"""
        SELECT 
            c.COLUMN_NAME AS 'Nama Kolom',
            c.COLUMN_TYPE AS 'Tipe Data MySQL',
            CONCAT(
                CASE WHEN c.IS_NULLABLE = 'YES' THEN '✅ NULL (Boleh Kosong)' ELSE '🛑 NOT NULL (Wajib Isi)' END,
                CASE WHEN c.EXTRA = 'auto_increment' THEN '\\n🚀 AUTO_INCREMENT' ELSE '' END
            ) AS 'Aturan Nullability & Increment',
            CASE 
                WHEN c.COLUMN_KEY = 'PRI' THEN '🔑 PRIMARY KEY (PK)'
                WHEN c.COLUMN_KEY = 'MUL' AND k.REFERENCED_TABLE_NAME IS NOT NULL THEN '🔗 FOREIGN KEY (FK)'
                WHEN c.COLUMN_KEY = 'MUL' THEN 'INDEX'
                ELSE '-'
            END AS 'Status Kunci',
            CASE 
                WHEN k.REFERENCED_TABLE_NAME IS NOT NULL THEN CONCAT(k.REFERENCED_TABLE_NAME, ' (', k.REFERENCED_COLUMN_NAME, ')')
                ELSE '-'
            END AS 'Rujukan Induk (FK Origin)',
            CASE WHEN c.DATA_TYPE = 'enum' THEN REPLACE(REPLACE(REPLACE(c.COLUMN_TYPE, 'enum(', ''), ')', ''), "'", "") ELSE '-' END AS 'Daftar Pilihan ENUM'
        FROM INFORMATION_SCHEMA.COLUMNS c
        LEFT JOIN INFORMATION_SCHEMA.KEY_COLUMN_USAGE k 
            ON c.TABLE_SCHEMA = k.TABLE_SCHEMA AND c.TABLE_NAME = k.TABLE_NAME AND c.COLUMN_NAME = k.COLUMN_NAME
        WHERE c.TABLE_SCHEMA = '{config['db_future']['database']}' AND c.TABLE_NAME = '{table}'
        ORDER BY c.ORDINAL_POSITION;
        """
        df_struct_future = pd.read_sql(query_future_struct, db_future).drop_duplicates(subset=['Nama Kolom'], keep='first').reset_index(drop=True)
        
        # Lacak tabel anak yang nge-FK ke PK tabel ini di DB_FUTURE
        pk_referenced_list_future = []
        for idx, row_skri in df_struct_future.iterrows():
            if row_skri['Status Kunci'] == '🔑 PRIMARY KEY (PK)':
                lookup_fk_query_future = f"""
                SELECT CONCAT(TABLE_NAME, ' (', COLUMN_NAME, ')') as relasi
                FROM INFORMATION_SCHEMA.KEY_COLUMN_USAGE
                WHERE REFERENCED_TABLE_SCHEMA = '{config['db_future']['database']}'
                  AND REFERENCED_TABLE_NAME = '{table}'
                  AND REFERENCED_COLUMN_NAME = '{row_skri['Nama Kolom']}';
                """
                df_relasi_luar_future = pd.read_sql(lookup_fk_query_future, db_future)
                if not df_relasi_luar_future.empty:
                    pk_referenced_list_future.append("\n".join(df_relasi_luar_future['relasi'].tolist()))
                else:
                    pk_referenced_list_future.append("-")
            else:
                pk_referenced_list_future.append("-")
        df_struct_future['Tabel Yang nge-FK (DB_FUTURE)'] = pk_referenced_list_future

        # 4. Deep Comparison Kesamaan Jeroan Kolom dasar
        cols_to_compare = ['Nama Kolom', 'Tipe Data MySQL', 'Aturan Nullability & Increment', 'Status Kunci', 'Rujukan Induk (FK Origin)', 'Daftar Pilihan ENUM']
        
        is_structure_identical = False
        if not df_struct_new.empty and not df_struct_future.empty:
            is_structure_identical = df_struct_new[cols_to_compare].equals(df_struct_future[cols_to_compare])

        # Wadah paket data tabel untuk di-render nanti
        table_package = {
            'name': table,
            'df_real_data': df_real_data,
            'df_struct_new': df_struct_new,
            'df_struct_future': df_struct_future,
            'is_identical': is_structure_identical
        }

        # 🔥 FILTER SAKTI CIMUT: Pisahkan antrean, utamakan yang bermasalah (revisi) ke atas!
        if is_structure_identical:
            identical_tables_queue.append(table_package)
        else:
            revisi_tables_queue.append(table_package)
            
    except Exception as e:
        print(f"❌ Gagal menganalisis awal tabel `{table}`: {e}")

# --- TAHAP B: MULAI PEN TAMPILAN VISUALISASI BERDASARKAN ANT REAN PRIORITAS ---

# 🚨 1. KELOMPOK UTAMA (PALING ATAS): DAFTAR TABEL YANG WAJIB DIREVISI 🚨
if revisi_tables_queue:
    print("\n" + "!"*80)
    print(f"🚨 [🔥 REVISI PRIORITY BOARD] TERDETEKSI {len(revisi_tables_queue)} TABEL BERBEDA - HARUS SEGERA DIPERBAIKI!")
    print("!"*80)
    
    for pkg in revisi_tables_queue:
        print(f"\n================================================================================")
        print(f"⚠️  [STATUS: TARGET REVISI] TABEL: {pkg['name'].upper()}")
        print(f"================================================================================")
        
        print(f"📋 1. Karakteristik Wadah Pandas Dataframe (.info()):")
        pkg['df_real_data'].info()
        print("\n" + "-"*60)
        
        print(f"🔍 2. [TABEL REAL SAAT INI] Arsitektur Fisik Kolom pada DB_NEW:")
        display(pkg['df_struct_new'].style.set_properties(**{'white-space': 'pre-wrap', 'text-align': 'left'}))
        print("\n" + "-"*60)
        
        print(f"🔮 3. [⚠️ TARGET BLUEPRINT RUJUKAN] Tinjau Struktur yang Benar pada DB_FUTURE di bawah:")
        if pkg['df_struct_future'].empty:
            print("❌ ERROR: Tabel ini tidak ditemukan / belum dibuat sama sekali di DB_FUTURE!")
        else:
            display(pkg['df_struct_future'].style.set_properties(**{'white-space': 'pre-wrap', 'text-align': 'left'}))
        print("\n" + "-"*60)
        
        print(f"📸 4. Sampel Isi Data Real di DB_NEW:")
        display(pkg['df_real_data'] if pkg['df_real_data'].empty else pkg['df_real_data'])
        print("\n" + "="*80)

# ✨ 2. KELOMPOK KEDUA (BAW AH): DAFTAR TABEL YANG SUDAH AMAN IDENTIK ✨
if identical_tables_queue:
    print("\n" + "="*80)
    print(f"✨ [🟢 PERFECTLY MATCHED BOARD] SEBANYAK {len(identical_tables_queue)} TABEL STRUKTURNYA SUDAH IDENTIK SAMA PERSIS!")
    print("="*80)
    
    for pkg in identical_tables_queue:
        print(f"\n================================================================================")
        print(f"✅ [STATUS: AMAN IDENTIK] TABEL: {pkg['name'].upper()}")
        print(f"================================================================================")
        
        print(f"📋 1. Karakteristik Wadah Pandas Dataframe (.info()):")
        pkg['df_real_data'].info()
        print("\n" + "-"*60)
        
        print(f"🔍 2. [TABEL REAL & FINAL] Arsitektur Fisik Kolom pada DB_NEW:")
        display(pkg['df_struct_new'].style.set_properties(**{'white-space': 'pre-wrap', 'text-align': 'left'}))
        print("\n" + "-"*60)
        
        print("✨ INDICATOR: [ DB_NEW ] & [ DB_FUTURE ] STRUKTUR SUDAH IDENTIK SAMA PERSIS 100% ✨")
        print("ℹ️  Tabel DB_FUTURE otomatis disembunyikan untuk kenyamanan pemandangan layar monitor.")
        print("\n" + "-"*60)
        
        print(f"📸 3. Sampel Isi Data Real di DB_NEW:")
        display(pkg['df_real_data'] if pkg['df_real_data'].empty else pkg['df_real_data'])
        print("\n" + "="*80)

 🕵️‍♂️ FORENSIK ARSITEKTUR KONTROL: DETEKTOR AUTO-SMART COMPILER (REVISI PRIORITY) 🕵️‍♂️ 

✨ [🟢 PERFECTLY MATCHED BOARD] SEBANYAK 21 TABEL STRUKTURNYA SUDAH IDENTIK SAMA PERSIS!

✅ [STATUS: AMAN IDENTIK] TABEL: IZIN_KARYAWAN
📋 1. Karakteristik Wadah Pandas Dataframe (.info()):
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1914 entries, 0 to 1913
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype          
---  ------            --------------  -----          
 0   id_izin           1914 non-null   int64          
 1   id_karyawan       1914 non-null   int64          
 2   jenis_izin        1914 non-null   object         
 3   tanggal_mulai     1914 non-null   object         
 4   tanggal_selesai   1914 non-null   object         
 5   waktu_mulai       1914 non-null   timedelta64[ns]
 6   waktu_selesai     1914 non-null   timedelta64[ns]
 7   keterangan_izin   1914 non-null   object         
 8   dokumen_lampiran  1914 non-null   object         
 9   creat

,Nama Kolom,Tipe Data MySQL,Aturan Nullability & Increment,Status Kunci,Rujukan Induk (FK Origin),Daftar Pilihan ENUM,Tabel Yang nge-FK (DB_NEW)
0,id_izin,bigint(20) unsigned,🛑 NOT NULL (Wajib Isi) 🚀 AUTO_INCREMENT,🔑 PRIMARY KEY (PK),-,-,absensi (id_izin) verifikasi_izin (id_izin)
1,id_karyawan,bigint(20) unsigned,🛑 NOT NULL (Wajib Isi),🔗 FOREIGN KEY (FK),karyawan (id_karyawan),-,-
2,jenis_izin,"enum('Ijin','Ijin Darurat','Lembur','Sakit')",🛑 NOT NULL (Wajib Isi),-,-,"Ijin,Ijin Darurat,Lembur,Sakit",-
3,tanggal_mulai,date,🛑 NOT NULL (Wajib Isi),-,-,-,-
4,tanggal_selesai,date,🛑 NOT NULL (Wajib Isi),-,-,-,-
5,waktu_mulai,time,🛑 NOT NULL (Wajib Isi),-,-,-,-
6,waktu_selesai,time,🛑 NOT NULL (Wajib Isi),-,-,-,-
7,keterangan_izin,text,🛑 NOT NULL (Wajib Isi),-,-,-,-
8,dokumen_lampiran,varchar(255),🛑 NOT NULL (Wajib Isi),-,-,-,-
9,created_at,timestamp,🛑 NOT NULL (Wajib Isi),-,-,-,-



------------------------------------------------------------
✨ INDICATOR: [ DB_NEW ] & [ DB_FUTURE ] STRUKTUR SUDAH IDENTIK SAMA PERSIS 100% ✨
ℹ️  Tabel DB_FUTURE otomatis disembunyikan untuk kenyamanan pemandangan layar monitor.

------------------------------------------------------------
📸 3. Sampel Isi Data Real di DB_NEW:


,id_izin,id_karyawan,jenis_izin,tanggal_mulai,tanggal_selesai,waktu_mulai,waktu_selesai,keterangan_izin,dokumen_lampiran,created_at
0,1,4,,2023-11-10,2023-11-10,0 days 15:30:00,0 days 17:00:00,Al muslin ekskul,,2023-11-13 07:34:57
1,2,4,,2023-11-13,2023-11-13,0 days 07:00:00,0 days 08:30:00,pengganti Al muslim,,2023-11-13 07:35:47
2,3,5,,2023-11-26,2023-11-26,0 days 16:00:00,0 days 18:00:00,Mengganti 2 jam kerja Senin 27 November 2023 j...,1701093214_10eabbf62174540924e1.jpeg,2023-11-27 20:53:34
3,4,5,,2023-12-02,2023-12-02,0 days 09:00:00,0 days 13:00:00,"Mengganti 4 jam kerja Kamis, 30 November 2023 ...",1701093670_6ed710a3b721f5b1b08d.pdf,2023-11-27 21:01:10
4,5,2,,2023-11-29,2023-11-29,0 days 13:00:00,0 days 15:00:00,Les Coding agnes,,2023-11-29 13:24:43
...,...,...,...,...,...,...,...,...,...,...
1909,1910,11,Ijin,2026-02-27,2026-02-27,0 days 07:00:00,0 days 16:05:00,Terlambat,,2026-04-06 17:01:49
1910,1911,11,Ijin,2026-03-31,2026-03-31,0 days 07:00:00,0 days 17:15:00,Terlambat,,2026-04-06 17:02:36
1911,1912,4,Ijin,2026-04-08,2026-04-08,0 days 18:15:00,0 days 19:15:00,"ijin pulang lebih cepat karena mau ke bengkel,...",,2026-04-08 11:40:20
1912,1913,18,Lembur,2026-03-31,2026-03-31,0 days 09:17:00,0 days 10:05:00,"tabungan jam maret, 58 menit",1775649994_ebf653ce91e06066e37c.jpg,2026-04-08 19:06:34




✅ [STATUS: AMAN IDENTIK] TABEL: VERIFIKASI_IZIN
📋 1. Karakteristik Wadah Pandas Dataframe (.info()):
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 0 entries
Data columns (total 7 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   id_verifikasi_izin      0 non-null      object
 1   id_izin                 0 non-null      object
 2   status_verifikasi_izin  0 non-null      object
 3   catatan_verifikator     0 non-null      object
 4   status_baca             0 non-null      object
 5   id_division             0 non-null      object
 6   created_at              0 non-null      object
dtypes: object(7)
memory usage: 132.0+ bytes

------------------------------------------------------------
🔍 2. [TABEL REAL & FINAL] Arsitektur Fisik Kolom pada DB_NEW:


...

,id_sm,tanggal_daftar,alamat_domisili,nama_lengkap,nama_panggilan,jenis_kelamin,nama_instansi,tingkat_sekolah,pekerjaan_sm,tempat_lahir,tanggal_lahir,nomor_induk_sm,email_sm,wa_sm,status_keluar_sm,id_mitra,sertifikat_sm




✅ [STATUS: AMAN IDENTIK] TABEL: SISWA_MITRA_KELUAR
📋 1. Karakteristik Wadah Pandas Dataframe (.info()):
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 0 entries
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   id_sm_keluar       0 non-null      object
 1   id_sm              0 non-null      object
 2   alasan_keluar_sm   0 non-null      object
 3   tanggal_keluar_sm  0 non-null      object
dtypes: object(4)
memory usage: 132.0+ bytes

------------------------------------------------------------
🔍 2. [TABEL REAL & FINAL] Arsitektur Fisik Kolom pada DB_NEW:


,Nama Kolom,Tipe Data MySQL,Aturan Nullability & Increment,Status Kunci,Rujukan Induk (FK Origin),Daftar Pilihan ENUM,Tabel Yang nge-FK (DB_NEW)
0,id_sm_keluar,bigint(20) unsigned,🛑 NOT NULL (Wajib Isi) 🚀 AUTO_INCREMENT,🔑 PRIMARY KEY (PK),-,-,-
1,id_sm,bigint(20) unsigned,✅ NULL (Boleh Kosong),🔗 FOREIGN KEY (FK),siswa_mitra (id_sm),-,-
2,alasan_keluar_sm,text,🛑 NOT NULL (Wajib Isi),-,-,-,-
3,tanggal_keluar_sm,date,🛑 NOT NULL (Wajib Isi),-,-,-,-



------------------------------------------------------------
✨ INDICATOR: [ DB_NEW ] & [ DB_FUTURE ] STRUKTUR SUDAH IDENTIK SAMA PERSIS 100% ✨
ℹ️  Tabel DB_FUTURE otomatis disembunyikan untuk kenyamanan pemandangan layar monitor.

------------------------------------------------------------
📸 3. Sampel Isi Data Real di DB_NEW:


,id_sm_keluar,id_sm,alasan_keluar_sm,tanggal_keluar_sm
